# Sprint 1 — Data Cleaning

**Project:** Explainable Prediction of Formula 1 Race Outcomes
**Sprint 1 Lead:** Sharon

## Purpose
This notebook applies the cleaning decisions surfaced in `sprint1_eda.ipynb` and produces the modelling-ready dataset at `data/processed/f1_modern_cleaned.csv`.


## Cleaning Decisions (from EDA)
1. Join the seven raw tables into a single driver-race level dataframe.
2. Filter to `year >= 2000` (modern era).
3. Derive a `finished` boolean from `position` non-null.
4. Retain both `position` (null for DNFs) and `positionOrder` (always populated).
5. Verify no duplicate `(raceId, driverId)` rows.
6. Parse `date` to datetime.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

In [2]:

NA = ['\\N']

races = pd.read_csv(RAW_DIR / 'races.csv', na_values=NA)
results = pd.read_csv(RAW_DIR / 'results.csv', na_values=NA)
qualifying = pd.read_csv(RAW_DIR / 'qualifying.csv', na_values=NA)
drivers = pd.read_csv(RAW_DIR / 'drivers.csv', na_values=NA)
constructors = pd.read_csv(RAW_DIR / 'constructors.csv', na_values=NA)
circuits = pd.read_csv(RAW_DIR / 'circuits.csv', na_values=NA)
status = pd.read_csv(RAW_DIR / 'status.csv', na_values=NA)

print("Raw tables loaded.")

Raw tables loaded.


In [3]:
df = results.copy()

df = df.merge(
    races[['raceId', 'year', 'round', 'circuitId', 'name', 'date']]
        .rename(columns={'name': 'race_name'}),
    on='raceId', how='left'
)

df = df.merge(
    drivers[['driverId', 'driverRef', 'forename', 'surname', 'nationality']]
        .rename(columns={'nationality': 'driver_nationality'}),
    on='driverId', how='left'
)
df['driver_name'] = df['forename'] + ' ' + df['surname']

df = df.merge(
    constructors[['constructorId', 'name', 'nationality']]
        .rename(columns={'name': 'constructor_name', 'nationality': 'constructor_nationality'}),
    on='constructorId', how='left'
)

df = df.merge(
    circuits[['circuitId', 'name', 'location', 'country']]
        .rename(columns={'name': 'circuit_name', 'country': 'circuit_country'}),
    on='circuitId', how='left'
)

df = df.merge(status, on='statusId', how='left')

df = df.merge(
    qualifying[['raceId', 'driverId', 'position']]
        .rename(columns={'position': 'qualifying_position'}),
    on=['raceId', 'driverId'], how='left'
)

print(f"Joined dataframe shape: {df.shape}")

Joined dataframe shape: (26759, 35)


In [4]:
df = df[df['year'] >= 2000].copy()

df['finished'] = df['position'].notna()

df['date'] = pd.to_datetime(df['date'])

assert df.duplicated(subset=['raceId', 'driverId']).sum() == 0, \
    "Duplicate (raceId, driverId) rows found after cleaning — investigate."

print(f"After era filter and cleaning: {df.shape}")

After era filter and cleaning: (10079, 36)


In [5]:
# Convert position columns to numeric
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
df['qualifying_position'] = pd.to_numeric(df['qualifying_position'], errors='coerce')
df['laps'] = pd.to_numeric(df['laps'], errors='coerce')
df['points'] = pd.to_numeric(df['points'], errors='coerce')

In [6]:
keep = [
    'resultId', 'raceId', 'year', 'round', 'date', 'race_name',
    'circuit_name', 'circuit_country',
    'driverId', 'driver_name', 'driver_nationality',
    'constructorId', 'constructor_name', 'constructor_nationality',
    'grid', 'qualifying_position',
    'position', 'positionOrder', 'points', 'laps',
    'status', 'finished'
]

clean = df[keep].sort_values(['year', 'round', 'positionOrder']).reset_index(drop=True)
print(f"Final cleaned shape: {clean.shape}")
clean.head(5)

Final cleaned shape: (10079, 22)


,resultId,raceId,year,round,date,race_name,circuit_name,circuit_country,driverId,driver_name,...,constructor_name,constructor_nationality,grid,qualifying_position,position,positionOrder,points,laps,status,finished
0,2931,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,30,Michael Schumacher,...,Ferrari,Italian,3,3.0,1.0,1,10.0,58,Finished,True
1,2932,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,22,Rubens Barrichello,...,Ferrari,Italian,4,4.0,2.0,2,6.0,58,Finished,True
2,2933,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,23,Ralf Schumacher,...,Williams,British,11,11.0,3.0,3,4.0,58,Finished,True
3,2934,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,35,Jacques Villeneuve,...,BAR,British,8,8.0,4.0,4,3.0,58,Finished,True
4,2935,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,21,Giancarlo Fisichella,...,Benetton,Italian,9,9.0,5.0,5,2.0,58,Finished,True


In [7]:
output_path = PROCESSED_DIR / 'f1_modern_cleaned.csv'
clean.to_csv(output_path, index=False)
print(f"Wrote {clean.shape[0]} rows to {output_path}")

Wrote 10079 rows to ..\data\processed\f1_modern_cleaned.csv


In [8]:
verify = pd.read_csv(output_path)
print(f"Re-loaded shape: {verify.shape}")
print(f"\nColumn types:")
print(verify.dtypes)
print(f"\nFirst rows:")
verify.head(3)

Re-loaded shape: (10079, 22)

Column types:
resultId                     int64
raceId                       int64
year                         int64
round                        int64
date                           str
race_name                      str
circuit_name                   str
circuit_country                str
driverId                     int64
driver_name                    str
driver_nationality             str
constructorId                int64
constructor_name               str
constructor_nationality        str
grid                         int64
qualifying_position        float64
position                   float64
positionOrder                int64
points                     float64
laps                         int64
status                         str
finished                      bool
dtype: object

First rows:


,resultId,raceId,year,round,date,race_name,circuit_name,circuit_country,driverId,driver_name,...,constructor_name,constructor_nationality,grid,qualifying_position,position,positionOrder,points,laps,status,finished
0,2931,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,30,Michael Schumacher,...,Ferrari,Italian,3,3.0,1.0,1,10.0,58,Finished,True
1,2932,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,22,Rubens Barrichello,...,Ferrari,Italian,4,4.0,2.0,2,6.0,58,Finished,True
2,2933,158,2000,1,2000-03-12,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,23,Ralf Schumacher,...,Williams,British,11,11.0,3.0,3,4.0,58,Finished,True


In [9]:
# Final sanity checks on the cleaned output
print("Position null count (should be > 0 — these are the DNFs):", clean['position'].isna().sum())
print("PositionOrder null count (should be 0):", clean['positionOrder'].isna().sum())
print("Finished rate:", f"{clean['finished'].mean():.1%}")
print(f"\nDate range: {clean['date'].min()} to {clean['date'].max()}")
print(f"Number of unique races: {clean['raceId'].nunique()}")
print(f"Number of unique drivers: {clean['driverId'].nunique()}")
print(f"Number of unique constructors: {clean['constructorId'].nunique()}")

Position null count (should be > 0 — these are the DNFs): 2141
PositionOrder null count (should be 0): 0
Finished rate: 78.8%

Date range: 2000-03-12 00:00:00 to 2024-12-08 00:00:00
Number of unique races: 479
Number of unique drivers: 126
Number of unique constructors: 38
